# Imágenes de contenedor y registro.

Un **contenedor** empaqueta una aplicación junto con su entorno de ejecución, de modo que el mismo artefacto pueda correr en la máquina de quien desarrolla, en un *pipeline* y en el destino de despliegue. La receta de ese empaquetado es un archivo llamado ```Dockerfile```. El resultado de construirla es una **imagen**; un **registro** es el almacén desde el que se publica y se descarga esa imagen.

## Objetivos.

* Leer el ```Dockerfile``` de la raíz del repositorio y relacionar cada instrucción con el arranque de la *API* del cuaderno 14.
* Construir y ejecutar la imagen en local.
* Distinguir la construcción de comprobación que ya hace el *pipeline* de *CI* de la publicación en un registro, que se hace al desplegar.


## El ```Dockerfile```.

El ```Dockerfile``` (y el ```.dockerignore```) están en la **raíz** de este repositorio y construyen la *API* del directorio ```api/```. El contenido vigente es, en esencia:

```dockerfile
FROM python:3.12-slim

ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1 \
    PIP_NO_CACHE_DIR=1 \
    PORT=8080

WORKDIR /app

RUN useradd --create-home --uid 10001 --shell /usr/sbin/nologin appuser

COPY api/requirements.txt /app/requirements.txt
RUN pip install --no-cache-dir -r /app/requirements.txt

COPY api/ /app/
RUN chown -R appuser:appuser /app

USER appuser

EXPOSE 8080

CMD ["sh", "-c", "exec gunicorn --bind 0.0.0.0:${PORT} --workers 1 --threads 4 --timeout 0 wsgi:app"]
```

Puntos que conviene observar:

* La imagen base lleva una **versión fija** (```3.12-slim```), no la etiqueta ```latest```.
* Las dependencias de ejecución salen de ```api/requirements.txt``` (no de ```requirements-dev.txt```, que es solo para linter y pruebas).
* El proceso corre como un usuario sin privilegios (```appuser```), no como *root*.
* El arranque es *gunicorn* sobre ```wsgi:app```, en primer plano, con un solo *worker*.
* El puerto se configura con la variable de entorno ```PORT```, que *Cloud Run* también inyecta.
* El ```.dockerignore``` evita copiar entornos virtuales, bitácoras, cachés y secretos a la imagen.


## Construcción local.

En la raíz del repositorio, con *Docker* instalado:

```bash
docker build -t cd441-api .
docker run --rm -p 8080:8080 -e APP_VERSION=local cd441-api
```

La clave de *Flask* y las demás variables de entorno se documentan en ```api/.env.example```. En un despliegue se definen en el entorno del contenedor; no se incrustan en la imagen ni se versionan valores reales.

La comprobación es una petición *HTTP* (*Hypertext Transfer Protocol*) a la ruta de salud:

```bash
curl -s http://127.0.0.1:8080/health
```

La respuesta incluye ```"status": "ok"``` y el valor de ```APP_VERSION```. La etiqueta ```cd441-api``` es solo de trabajo en la máquina propia. En el *pipeline* de despliegue se usará una etiqueta ligada al *commit* y un registro compartido.


## Dónde se construye y dónde se publica.

El *workflow* ```.github/workflows/ci.yml``` ya incluye un paso ```docker build -t cd441-api:ci .``` **sin** ```docker push```. Su función es fallar pronto si el ```Dockerfile``` está roto. No deja una imagen en ningún registro.

La publicación ocurre en el cuaderno 16, al desplegar en *Cloud Run*: la imagen se envía a **Artifact Registry**, el registro nativo de *Google Cloud Platform* (*GCP*), con una etiqueta inmutable (el *SHA* del *commit*). *Cloud Run* consume de forma natural las imágenes de ese registro.

Otros registros, como *GitHub Container Registry* (*GHCR*, el registro de contenedores de *GitHub*, en ```ghcr.io```), siguen el mismo principio (etiqueta inmutable, no solo ```latest```). Este curso no publica en *GHCR*; se menciona para no confundirlo con *Artifact Registry*.

La regla que sí se lleva de aquí: **no reconstruir la imagen por entorno**. Se construye una vez, se etiqueta de forma única y se promociona o se despliega esa misma referencia.


## Actividad.

1. Construir la imagen en local y comprobar que ```GET /health``` responde.
2. Revisar en una ejecución del *workflow* ```CI``` el paso ```Construir imagen (sin publicar)``` y confirmar que no aparece un *push* a un registro.
3. Abrir el ```Dockerfile``` y señalar qué instrucción impide que el proceso corra como *root*, y cuál fija el puerto de escucha.

El cuaderno 16 publica esa misma imagen en *Artifact Registry* y la despliega en *Cloud Run*.


## Cierre.

La *API* ya no es solo un proyecto de *Python*: es una imagen con versión de base fija, usuario sin privilegios y un proceso de arranque explícito. Esa imagen es lo que se publicará y se desplegará, sin reconstruirla por entorno, en *Cloud Run* y en *Linux*.


<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2023.</p>